        # Exercise 4 - Monte Carlo uncertainty analysis

        [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea-workshop/blob/main/notebooks/04_monte_carlo_uncertainty.ipynb)

        Prices, capital estimates, and process quantities are uncertain, and they are uncertain **at the same time**.
Monte Carlo simulation samples all of them from probability distributions, recomputes the plant thousands of
times, and returns distributions of NPV, levelized cost, and the other metrics instead of single numbers.

**Time in the session: 15 minutes** (Exercise 4 of 6). Section 5 on parameter dependencies is a short demo:
skim it if you are behind.

        ## Learning goals

        - Attach uncertainty distributions to prices, process quantities, the operator rate, and project-level factors.
- Run `monte_carlo()` and plot the resulting distributions of LCOP and NPV.
- Extract percentiles and probabilities (for example P(NPV > 0)) from the raw samples.
- Check the sampled inputs, and compare two plants under uncertainty.
- Declare dependencies between parameters so that Monte Carlo, sensitivity, and tornado analyses respect the process.

        ## Background references

        - Tamarona, P.B., Vlugt, T.J.H., & Ramdin, M. (2026). OpenPyTEA: An open-source python toolkit for techno-economic assessment of chemical process plants and energy systems with economic sensitivity and uncertainty evaluation. *SoftwareX, 35*, 102816. https://doi.org/10.1016/j.softx.2026.102816
- Van der Spek, M., Roussanaly, S., & Rubin, E.S. (2019). Best practices and recent advances in CCS cost engineering and economic analysis. *IJGGC, 83*, 91-104. https://doi.org/10.1016/j.ijggc.2019.02.006
- OpenPyTEA documentation: https://openpytea.readthedocs.io

## 0) Setup: rebuild the green-ammonia plant

In [ ]:
# Run this cell first. It installs OpenPyTEA if it is missing (for example on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
from openpytea import Equipment

# Equipment of the green-ammonia synthesis loop (sizes from the process simulation, see notebook 01)
compressor  = Equipment(name="C-101 Syngas compressor", param=1_200, process_type="Fluids",
                        category="Compressors, fans, & blowers", type="Compressor, centrifugal",
                        material="Carbon steel")
motor       = Equipment(name="M-101 Compressor motor", param=1_200 / 0.90, process_type="Electrical",
                        category="Motors & generators", type="Totally enclosed motor")
exchanger   = Equipment(name="E-101 Feed/effluent exchanger", param=250, process_type="Fluids",
                        category="Heat Exchangers", type="U-tube shell & tube",
                        material="304 stainless steel")
converter   = Equipment(name="R-101 Ammonia converter", param=12_000, process_type="Fluids",
                        category="Reactors", type="Tubular fixed bed", material="304 stainless steel")
separator   = Equipment(name="V-101 Ammonia separator", param=25, process_type="Fluids",
                        category="Pressure vessels", type="Vertical", material="Carbon steel")
chiller     = Equipment(name="K-101 Refrigeration unit", param=800, process_type="Fluids",
                        category="Utilities", type="Packaged mechanical refrigerator")
product_pump = Equipment(name="P-101 Product pump", param=20, process_type="Fluids",
                         category="Pumps", type="Single-stage centrifugal", material="Carbon steel")

equipment_list = [compressor, motor, exchanger, converter, separator, chiller, product_pump]

In [ ]:
from openpytea import Plant

ammonia_config = {
    "plant_name": "Green Ammonia",
    "process_type": "Fluids",
    "country": "Netherlands",
    "equipment": equipment_list,

    # Financial assumptions
    "interest_rate": 0.08,
    "project_lifetime": 20,      # years
    "plant_utilization": 0.90,   # fraction of the year on stream
    "tax_rate": 0.25,

    # Products: the first entry is the main product used for the levelized cost
    "plant_products": {
        "ammonia": {"production": 100_000, "price": 0.95},   # kg/day, USD/kg
    },

    # Variable OPEX: daily consumption and unit price
    "variable_opex_inputs": {
        "hydrogen":    {"consumption": 17_800, "price": 2.00},   # kg/day,  USD/kg
        "nitrogen":    {"consumption": 82_400, "price": 0.05},   # kg/day,  USD/kg
        "electricity": {"consumption": 60_000, "price": 0.08},   # kWh/day, USD/kWh
    },
}

ammonia_plant = Plant(ammonia_config)
ammonia_plant.calculate_all()
print(f"Fixed capital investment : {ammonia_plant.fixed_capital:,.0f} USD")
print(f"Levelized cost of ammonia: {ammonia_plant.levelized_cost:.3f} USD/kg")
print(f"Net present value        : {ammonia_plant.npv:,.0f} USD")

In [ ]:
import numpy as np
from copy import deepcopy
from openpytea.analysis import monte_carlo, tornado_data, sensitivity_data
from openpytea.plotting import (plot_monte_carlo, plot_multiple_monte_carlo, plot_monte_carlo_inputs,
                                plot_tornado, plot_sensitivity)

## 1) Describing what you do not know

Uncertainty is configured **per item**, in the same configuration dictionary:

| Where | Key | Samples |
|---|---|---|
| any `variable_opex_inputs` or `plant_products` entry | `price_uncertainty` | the price |
| any `variable_opex_inputs` entry | `consumption_uncertainty` | the daily consumption |
| any `plant_products` entry | `production_uncertainty` | the daily production |
| `operator_hourly_rate` | `rate_uncertainty` | the operator wage |
| top level | `project_uncertainties` | `fixed_capital_factor`, `fixed_opex_factor`, `project_lifetime`, `interest_rate`, `plant_utilization`, `tax_rate` |

Each uncertainty block takes `std`, `min`, and `max` (a **truncated normal** around the base value by default), or
a `dist_id` to pick another `scipy.stats` family: 2 lognormal, 3 normal, 4 uniform, 5 triangular, 7 discrete
uniform, 9 gamma, 10 beta, 12 Student's t, and more (see the Analysis user guide). The four project-level factors
`fixed_capital_factor` (std 0.3), `fixed_opex_factor` (std 0.3), `project_lifetime` (std 5 years), and
`interest_rate` (std 0.03) are sampled **by default**; set `"std": 0` to freeze one.

In [ ]:
ammonia_plant.update_configuration({
    "plant_products": {
        "ammonia": {"price_uncertainty": {"dist_id": 5, "min": 0.70, "max": 1.25}},            # triangular, mode at the base price 0.95
    },
    "variable_opex_inputs": {
        "hydrogen":    {"price_uncertainty": {"std": 0.50, "min": 1.0, "max": 4.0}},          # truncated normal
        "electricity": {"price_uncertainty": {"std": 0.03, "min": 0.02, "max": 0.20}},
        "nitrogen":    {"price_uncertainty": {"std": 0.01, "min": 0.02, "max": 0.10}},
    },
    "operator_hourly_rate": {"rate": 38.11, "rate_uncertainty": {"std": 8, "min": 25, "max": 60}},
    "project_uncertainties": {
        "fixed_capital_factor": {"std": 0.30, "min": 0.6, "max": 1.6},   # +-30 % class-4 estimate accuracy
        "fixed_opex_factor":    {"std": 0.15, "min": 0.7, "max": 1.3},
        "interest_rate":        {"std": 0.015, "min": 0.04, "max": 0.12},
        "project_lifetime":     {"std": 0},                              # frozen at 20 years
        "plant_utilization":    {"dist_id": 4, "min": 0.80, "max": 0.95},  # uniform, opt-in
    },
})

## 2) Running the simulation

20 000 samples take a couple of seconds for this plant. Increase `num_samples` for smoother histograms;
`batch_size` only controls memory use. `random_seed` makes the run reproducible.

In [ ]:
mc_base = monte_carlo(ammonia_plant, num_samples=20_000, batch_size=5_000, random_seed=2026)

fig, ax = plot_monte_carlo(mc_base, metric="LCOP", bins=40, label="Levelized cost of ammonia / [USD/kg]")
fig, ax = plot_monte_carlo(mc_base, metric="NPV", bins=40)

The result is a dictionary with the raw samples under `"metrics"` (`LCOP`, `NPV`, `ROI`, `PBT`) and the sampled
inputs under `"inputs"`. Percentiles and probabilities are one line of numpy away:

In [ ]:
lcop = np.asarray(mc_base["metrics"]["LCOP"])
npv = np.asarray(mc_base["metrics"]["NPV"])

p10, p50, p90 = np.percentile(lcop, [10, 50, 90])
print(f"LCOP  P10 / P50 / P90 : {p10:.3f} / {p50:.3f} / {p90:.3f} USD/kg   (deterministic: {ammonia_plant.levelized_cost:.3f})")
print(f"P(NPV > 0)            : {np.mean(npv > 0):.1%}")
print(f"P(LCOP < 0.95 USD/kg) : {np.mean(lcop < 0.95):.1%}")

## 3) Always look at the inputs

`plot_monte_carlo_inputs()` shows the sampled distributions, split into **process** quantities and **economic**
parameters. Inputs without an uncertainty block appear as a single bar at their fixed value, so you see what
varies and what does not.

In [ ]:
fig, axes = plot_monte_carlo_inputs(mc_base, category="economic", bins=40)

## 4) Comparing two plants under uncertainty

A deterministic comparison gives one number per design; Monte Carlo tells you how often the ranking holds.
Let us compare the base case with a design that pays about 7 % more capital for a converter with 5 % lower
hydrogen consumption. Deterministically the two levelized costs differ by less than 0.01 USD/kg.

In [ ]:
efficient = deepcopy(ammonia_plant)
efficient.update_configuration({
    "plant_name": "Efficient converter",
    "fixed_capital_factors": {"contingency": 0.20},                         # contingency 10 % -> 20 %, about +7 % FCI
    "variable_opex_inputs": {"hydrogen": {"consumption": 17_800 * 0.95}},
})
efficient.calculate_all()
mc_efficient = monte_carlo(efficient, num_samples=20_000, batch_size=5_000, random_seed=2026)

fig, ax = plot_multiple_monte_carlo([mc_base, mc_efficient], metric="LCOP", bins=40,
                                    label="Levelized cost of ammonia / [USD/kg]")
print(f"Deterministic LCOP: {ammonia_plant.levelized_cost:.3f} vs {efficient.levelized_cost:.3f} USD/kg")
print(f"P(efficient design is cheaper): {np.mean(np.asarray(mc_efficient['metrics']['LCOP']) < lcop):.1%}")

## 5) Going further: dependencies between parameters (short demo)

So far every input was sampled independently. In reality the hydrogen and nitrogen consumption follow the
ammonia production (stoichiometry), and the electricity consumption follows it too. OpenPyTEA lets you declare
such links as a small graph: `dependent = Σ wᵢ · parentᵢ + offset`. Because the dependency lives on the plant,
Monte Carlo, `sensitivity_data()`, and `tornado_data()` all honour it, and a dependent can no longer be varied on
its own.

In [ ]:
ammonia_plant.update_configuration({
    "plant_products": {
        "ammonia": {"production_uncertainty": {"std": 5_000, "min": 85_000, "max": 110_000}},   # the root of the graph
    },
    "variable_opex_inputs": {
        "hydrogen":    {"consumption_dependency": {"depends_on": {"production:ammonia": 17_800 / 100_000}}},
        "nitrogen":    {"consumption_dependency": {"depends_on": {"production:ammonia": 82_400 / 100_000}}},
        "electricity": {"consumption_dependency": {"depends_on": {"production:ammonia": 60_000 / 100_000}}},
    },
})

mc_linked = monte_carlo(ammonia_plant, num_samples=20_000, batch_size=5_000, random_seed=2026)
fig, axes = plot_monte_carlo_inputs(mc_linked, category="process", bins=40)

In [ ]:
# The tornado now sweeps ammonia production together with the feeds it drives, and drops the dependents as separate bars
fig, ax = plot_tornado(tornado_data(ammonia_plant, plus_minus_value=0.3, metric="NPV", include_process_params=True))

## 6) Task 4.1: how much would better cost data be worth?

The capital cost estimate is the largest single uncertainty (`fixed_capital_factor`, ± 30 %). Suppose a detailed
engineering study could narrow it to ± 10 %.

1. Copy `ammonia_plant`, set `project_uncertainties.fixed_capital_factor` to `{"std": 0.10, "min": 0.8, "max": 1.2}`,
   and rerun Monte Carlo with the same seed.
2. Compare P10 / P90 of the NPV and P(NPV > 0) before and after. How much of the NPV spread was due to the capital estimate?
3. Bonus: repeat for the hydrogen price (`std` 0.5 to 0.2). Which piece of information would you buy first?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def summarize(mc, name):
    npv = np.asarray(mc["metrics"]["NPV"])
    p10, p90 = np.percentile(npv, [10, 90])
    print(f"{name:<28} NPV P10 {p10/1e6:>7.1f}  P90 {p90/1e6:>6.1f} MUSD   spread {(p90-p10)/1e6:>6.1f} MUSD   P(NPV>0) {np.mean(npv>0):.1%}")

summarize(mc_linked, "base uncertainties")

better_capex = deepcopy(ammonia_plant)
better_capex.update_configuration({"project_uncertainties": {"fixed_capital_factor": {"std": 0.10, "min": 0.8, "max": 1.2}}})
summarize(monte_carlo(better_capex, num_samples=20_000, batch_size=5_000, random_seed=2026), "capex +-10 %")

better_h2 = deepcopy(ammonia_plant)
better_h2.update_configuration({"variable_opex_inputs": {"hydrogen": {"price_uncertainty": {"std": 0.20, "min": 1.0, "max": 4.0}}}})
summarize(monte_carlo(better_h2, num_samples=20_000, batch_size=5_000, random_seed=2026), "hydrogen price std 0.2")
```

Both refinements narrow the NPV spread by roughly 10 %, and neither changes P(NPV > 0) much: the remaining spread
comes from the ammonia price and the other inputs sampled at the same time. Narrowing the hydrogen price helps
slightly more than narrowing the capital estimate, so a long-term hydrogen supply contract is the first thing to
secure. Note that the hydrogen and ammonia prices are sampled independently here; in reality they are
correlated, which is exactly the kind of link the dependency mechanism above can express.
</details>

## Recap

- Uncertainty blocks (`price_uncertainty`, `consumption_uncertainty`, `production_uncertainty`, `rate_uncertainty`, `project_uncertainties`) live next to the values they describe.
- `monte_carlo()` returns raw samples; percentiles and probabilities are yours to compute.
- Look at the inputs, compare plants with `plot_multiple_monte_carlo()`, and declare dependencies so the sweeps stay physical.

**Next:** [Exercise 5 - Case study: hydrogen production pathways](05_case_study_hydrogen.ipynb)